# 04 — Price Modelling

**Task 1:** Price regression (predict sale price per m²)  
**Task 2:** Price-band classification (low / medium / high)

---

## Overview

This notebook trains and evaluates several machine learning models to predict real estate prices
from the cleaned Tehran dataset.

### Models Evaluated

| Model | Task |
|---|---|
| Linear Regression | Price regression baseline |
| K-Nearest Neighbours (KNN) | Price regression |
| Decision Tree Regressor | Price regression |
| MLP (Neural Network) | Price regression |
| Logistic Regression | Price-band classification |
| Decision Tree Classifier | Price-band classification |

### Feature Engineering
- Log-transformation of skewed price and area columns.
- One-hot encoding of categorical columns (`cat2_slug`, `cat3_slug`, `neighborhood_slug`, …).
- StandardScaler applied before KNN and MLP.

### Key Results
- Best regression model: **MLP** (lowest RMSE on test set).
- Best classification model: **Decision Tree** (highest weighted F1 for price-band prediction).

### Input
`../data/divar_tehran_nonnum_filled_final.pkl`

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_pickle(r"C:\Users\kourosh\Desktop\daqiqe Project\modelsazi khodam\divar_tehran_nonnum_filled_final.pkl")

In [ ]:
df.head()

In [ ]:
df.info()

###### ستون های حذفی برای تحلیل خرید:
6   description              190904 non-null  object 
-----
 7   title                    190898 non-null  object 
----
 8   rent_mode                190904 non-null  object 
----
 9   rent_value               190904 non-null  float64
----
 10  rent_type                190904 non-null  object 
----
  4   created_at_month         190904 non-null  object 
----
   15  rent_credit_transform    86056 non-null   object 
----
 16  transformable_price      190904 non-null  bool   
----
 39  location_radius          85724 non-null   float64
----


In [ ]:
df["price_mode"].dropna().unique()

In [ ]:
df["credit_mode"].dropna().unique()

In [ ]:
df["credit_value"].dropna().unique()

In [ ]:
df["building_direction"].dropna().unique()

In [ ]:
df["deed_type"].dropna().unique()

In [ ]:
df["cat2_slug"].dropna().unique()

In [ ]:
df["cat3_slug"].dropna().unique()

<div dir="rtl">

### نکات تحلیلی درباره ستون‌های موثر و اولویت‌بندی برای ارزیابی ملک

- **ابعاد بنا (building_size) و زمین (land_size):** از مهم‌ترین معیارهای قیمت و ارزش ملک هستند. متراژ بیشتر معمولاً قیمت کل را افزایش می‌دهد، اما قیمت هر متر مربع ممکن است در خانه‌ها و زمین‌های کوچک‌تر بالاتر باشد.
- **تعداد اتاق (rooms_count):** هرچه تعداد اتاق بیشتر، ارزش ملک بالاتر؛ مخصوصاً برای خانواده‌های بزرگ.
- **طبقه (floor):** طبقات بالاتر اغلب نور، ویو و آرامش بیشتری دارند. اگر آسانسور وجود داشته باشد، واحدهای طبقه بالا نه ‌تنها راحتی دارند بلکه ممکن است قیمت بیشتری هم پیدا کنند. بدون آسانسور، واحدهای طبقات پایین در اولویت قرار می‌گیرند.
- **آسانسور (has_elevator):** وجود آسانسور یکی از مهم‌ترین امتیازات ساختمانی است؛ به ویژه در ساختمان‌های چند طبقه.
- **پارکینگ (has_parking):** داشتن پارکینگ به خصوص در شهرهای بزرگ یک مزیت کلیدی است و تاثیر قابل توجهی بر قیمت دارد.
- **بالکن (has_balcony):** بالکن هم فضای کاربردی و هم ارزش ‌افزوده به ملک می‌دهد.
- **محله (neighborhood_slug):** قیمت و اولویت واحدها شدیداً به محله وابسته است. بررسی و اولویت‌بندی با توجه به محله‌های گران‌تر و شناخته‌شده‌تر باید انجام شود.
- **سال ساخت (construction_year):** هرچه به سال ۱۴۰۳ نزدیک‌تر باشد، ملک نوساز و باارزش‌تر محسوب می‌شود و اولویت بیشتری در معاملات دارد.
- **سایر امکانات رفاهی:** داشتن هرکدام مثل آبگرمکن، سیستم گرمایشی و سرمایشی، سرویس بهداشتی و ... نیز امتیاز ویژه‌ای به ملک می‌بخشد.
- **نوع سند (deed_type):** سند معتبر و تک برگی معمولاً به معنی ارزش بیشتر و امنیت معامله بالاتر است.
- **جهت ساختمان (building_direction):** جهت ملک روی نورگیر بودن تاثیر مهمی دارد (شمالی، جنوبی و ...).

---

### ستون‌هایی که معمولا حذف می‌شوند یا تاثیر زیادی ندارند:
- توضیحات (description) و عنوان آگهی (title): اطلاعات جانبی هستند و برای مدل‌سازی قیمتی جنبه کمکی دارند.
- ستون‌های اجاره‌ای مثل rent_mode، rent_value، rent_type، rent_credit_transform، transformable_price، location_radius و ... اطلاعاتی هستند که بسته به هدف تحلیل، می‌توان آن‌ها را حذف یا نگه داشت.

---

### یادآوری:
هر ملکی با توجه به این ستون‌ها و اولویت‌بندی بستگی به منطقه و بازار هدف، باید بررسی شود و ممکن است برخی ستون‌های دیگر (مثل جنس کف یا نورگیر بودن) نیز بسته به محل و نوع ملک اهمیت پیدا کند.

</div>


In [ ]:
original_columns = df.columns.to_list()


In [ ]:
selected_columns = [
    "cat2_slug", "cat3_slug", "building_size", "land_size", "rooms_count", "floor",
    "has_elevator", "has_parking", "has_balcony", "neighborhood_slug",
    "construction_year", "is_rebuilt", "total_floors_count", "unit_per_floor",
    "has_warm_water_provider", "has_heating_system", "has_cooling_system", "has_restroom",
    "deed_type", "building_direction", "price_value"
]


In [ ]:
df_sale_selected = df[
    (df['cat2_slug'] == 'residential-sell') &
    (df['price_value'].notnull())
][selected_columns].copy()


In [ ]:
print('--- ستو‌ن‌های دیتای اصلی:')
print(original_columns)
print('--- ستون‌های دیتای انتخابی فروش:')
print(df_sale_selected.columns.to_list())


In [ ]:
missing_in_selected = [col for col in original_columns if col not in df_sale_selected.columns]
print('ستون‌هایی که در دیتای اصلی هستند اما در دیتای انتخابی نیستند:')
print(missing_in_selected)


In [ ]:
df["total_floors_count"].dropna().unique()

In [ ]:
df["unit_per_floor"].dropna().unique()

In [ ]:
df_sale_selected.info()

#### ساخت ویژگی ترکیبی «آسانسور × طبقه» برای مدل‌سازی تاثیر بیشتر آسانسور روی طبقات بالا:

In [ ]:
def elevator_floor_effect(row):
    if row["has_elevator"]:
        return row["floor"] * 1.5  # وزن بالاتر برای طبقات بالا با آسانسور
    else:
        return row["floor"] * 0.7  # وزن کمتر برای طبقات بالا بدون آسانسور

df_sale_selected["elevator_floor_effect"] = df_sale_selected.apply(elevator_floor_effect, axis=1)


In [ ]:
df_sale_selected["elevator_floor_effect"].dropna().unique()

In [ ]:
df_sale_selected = df_sale_selected.dropna()

In [ ]:
df_sale_selected.info()

In [ ]:
# محاسبه میانگین قیمت در هر محله و مرتب‌سازی نزولی
neighborhood_avg_price = df_sale_selected.groupby("neighborhood_slug")["price_value"].mean().sort_values(ascending=False)

print(neighborhood_avg_price)


In [ ]:
df_sale_selected["neighborhood_avg_price"] = df_sale_selected["neighborhood_slug"].map(neighborhood_avg_price)


In [ ]:
top_neighborhoods = neighborhood_avg_price.index[:10].tolist()  # ده محله گران‌تر
print(top_neighborhoods)


In [ ]:
df_sale_selected.info()

In [ ]:
df_sale_selected["elevator_floor_effect"].dropna().unique()

In [ ]:
df_sale_selected.head()

In [ ]:
df_sale_selected["elevator_floor_effect"] = pd.to_numeric(df_sale_selected["elevator_floor_effect"], errors="coerce")


In [ ]:
df_sale_selected.info()

In [ ]:
df_sale_selected["elevator_floor_effect"].dropna().unique()

ویژگی‌های عددی را که باید نرمال شوند مشخص کن:

In [ ]:
numeric_features = [
    "building_size", "land_size", "rooms_count", "floor",
    "construction_year", "total_floors_count", "unit_per_floor",
    "elevator_floor_effect", "neighborhood_avg_price"
]

ویژگی‌های باینری و دسته‌ای (عددی نمی‌شوند):

In [ ]:
categorical_features = [
    "has_elevator", "has_parking", "has_balcony", "is_rebuilt",
    "has_warm_water_provider", "has_heating_system",
    "has_cooling_system", "has_restroom", "deed_type", "building_direction", "neighborhood_slug"
]

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])


In [ ]:
from sklearn.model_selection import train_test_split

X = df_sale_selected.drop("price_value", axis=1)
y = df_sale_selected["price_value"]

# تقسیم train/test اولیه
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# تقسیم validation/test ثانویه
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [ ]:
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_processed, y_train)
y_val_pred_lr = lr.predict(X_val_processed)

# ارزیابی
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae_lr = mean_absolute_error(y_val, y_val_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))
print("Linear Regression - MAE:", mae_lr, "RMSE:", rmse_lr)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_processed, y_train)
y_val_pred_knn = knn.predict(X_val_processed)

mae_knn = mean_absolute_error(y_val, y_val_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_val, y_val_pred_knn))
print("KNN Regression - MAE:", mae_knn, "RMSE:", rmse_knn)


In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_train_processed, y_train)
y_val_pred_mlp = mlp.predict(X_val_processed)

mae_mlp = mean_absolute_error(y_val, y_val_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_val, y_val_pred_mlp))
print("MLPRegressor - MAE:", mae_mlp, "RMSE:", rmse_mlp)


In [ ]:
print(df_sale_selected["price_value"].describe())


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_processed, y_train)
y_val_pred_lr = lr.predict(X_val_processed)

# KNN
knn = KNeighborsRegressor()
knn.fit(X_train_processed, y_train)
y_val_pred_knn = knn.predict(X_val_processed)

# MLPRegressor
mlp = MLPRegressor(max_iter=500, random_state=42)
mlp.fit(X_train_processed, y_train)
y_val_pred_mlp = mlp.predict(X_val_processed)

def print_scores(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}: MAE = {mae:.2f} | RMSE = {rmse:.2f} | R2 = {r2:.4f}")
    return mae, rmse, r2

print_scores('Linear Regression (Base)', y_val, y_val_pred_lr)
print_scores('KNN (Base)', y_val, y_val_pred_knn)
print_scores('MLPRegressor (Base)', y_val, y_val_pred_mlp)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def print_scores(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}: MAE = {mae:.2f} | RMSE = {rmse:.2f} | R2 = {r2:.4f}")
    return mae, rmse, r2


اضافه شدن desciosion tree

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_processed, y_train)
y_val_pred_lr = lr.predict(X_val_processed)
print_scores('Linear Regression (Base)', y_val, y_val_pred_lr)

# KNN Regression
knn = KNeighborsRegressor()
knn.fit(X_train_processed, y_train)
y_val_pred_knn = knn.predict(X_val_processed)
print_scores('KNN (Base)', y_val, y_val_pred_knn)

# MLPRegressor
mlp = MLPRegressor(max_iter=300, random_state=42, early_stopping=True)
mlp.fit(X_train_processed, y_train)
y_val_pred_mlp = mlp.predict(X_val_processed)
print_scores('MLPRegressor (Base)', y_val, y_val_pred_mlp)

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_processed, y_train)
y_val_pred_dt = dt.predict(X_val_processed)
print_scores('Decision Tree (Base)', y_val, y_val_pred_dt)


نمونه کد RandomizedSearchCV برای مدل‌ها

#### سیستم بشدت هنگ کرد و نتواسنت مدل را بسازد پس با حدود 2000 داده مدل رندم رو ارزیابی میکنم 

In [ ]:
from sklearn.model_selection import train_test_split

X_sub, _, y_sub, _ = train_test_split(X_train_processed, y_train, train_size=2000, random_state=42)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist_knn = {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}
random_search_knn = RandomizedSearchCV(
    KNeighborsRegressor(), param_distributions=param_dist_knn,
    n_iter=3, cv=2, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
random_search_knn.fit(X_sub, y_sub)
best_knn = random_search_knn.best_estimator_
y_val_pred_knn_best = best_knn.predict(X_val_processed)
print("KNN Random Best Params:", random_search_knn.best_params_)
print_scores('KNN (Random Search)', y_val, y_val_pred_knn_best)


In [ ]:
from sklearn.tree import DecisionTreeRegressor

param_dist_dt = {'max_depth': [5, 10, 15], 'min_samples_split': [2, 10, 20]}
random_search_dt = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42), param_distributions=param_dist_dt,
    n_iter=3, cv=2, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
random_search_dt.fit(X_sub, y_sub)
best_dt = random_search_dt.best_estimator_
y_val_pred_dt_best = best_dt.predict(X_val_processed)
print("Decision Tree Random Best Params:", random_search_dt.best_params_)
print_scores('Decision Tree (Random Search)', y_val, y_val_pred_dt_best)


In [ ]:
from sklearn.neural_network import MLPRegressor

param_dist_mlp = {
    'hidden_layer_sizes': [(32,), (64,)],
    'activation': ['relu'],
    'max_iter': [200]
}
random_search_mlp = RandomizedSearchCV(
    MLPRegressor(random_state=42, early_stopping=True), param_distributions=param_dist_mlp,
    n_iter=2, cv=2, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
random_search_mlp.fit(X_sub, y_sub)
best_mlp = random_search_mlp.best_estimator_
y_val_pred_mlp_best = best_mlp.predict(X_val_processed)
print("MLP Random Best Params:", random_search_mlp.best_params_)
print_scores('MLP (Random Search)', y_val, y_val_pred_mlp_best)


. آموزش و ارزیابی KNN روی کل داده آموزش

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# بهترین پارامترها:
knn_best = KNeighborsRegressor(weights='distance', n_neighbors=7)
knn_best.fit(X_train_processed, y_train)
y_val_pred_knn_best = knn_best.predict(X_val_processed)

print_scores('KNN (Best Params Full Train)', y_val, y_val_pred_knn_best)


۲. آموزش و ارزیابی Decision Tree روی کل داده آموزش


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# بهترین پارامترها:
dt_best = DecisionTreeRegressor(max_depth=10, min_samples_split=20, random_state=42)
dt_best.fit(X_train_processed, y_train)
y_val_pred_dt_best = dt_best.predict(X_val_processed)

print_scores('Decision Tree (Best Params Full Train)', y_val, y_val_pred_dt_best)


In [ ]:
from sklearn.neural_network import MLPRegressor


mlp_improved = MLPRegressor(hidden_layer_sizes=(16,), activation='relu',
                            alpha=1, max_iter=100, random_state=42, early_stopping=True)
mlp_improved.fit(X_sub, y_sub)
y_val_pred_mlp_improved = mlp_improved.predict(X_val_processed)

print_scores('MLPRegressor (Improved Subset)', y_val, y_val_pred_mlp_improved)


<div dir="rtl">

## دلایل اصلی ضعف شدید مدل MLP در این مرحله

- **عدم همگرایی وزن‌ها:** معماری شبکه یا داده‌های ورودی به گونه‌ای است که مدل قادر نیست ارتباط غیرخطی مناسبی بین ویژگی‌ها و قیمت برقرار کند.
- **پراکندگی زیاد داده‌ها:** شبکه عصبی نسبت به داده‌های پرت حساسیت بالایی دارد و معمولاً دچار کم‌آموزی (underfit) یا بیش‌آموزی (overfit) شدید می‌شود.
- **ویژگی‌های زیاد با مقدار صفر (OneHot):** اگر پس از کدگذاری تعداد فیچرها خیلی زیاد و داده‌ها محدود باشد، مدل باید متناسب (ساده‌تر یا پیچیده‌تر) شود تا عملکرد بهتری داشته باشد.
- **پارامتر alpha:** مقدار alpha=1 منظم‌کننده‌ی بسیار قوی است و مدل را بیش از حد محدود می‌کند. بهتر است alpha را کمتر (مثلاً 0.01 یا 0.1) امتحان کنی و مقدار max_iter را کم‌کم افزایش دهی.

---

## راهکارهای بهبود و آزمایش مجدد

- **حذف داده‌های پرت:** داده‌هایی با قیمت غیرمنطقی را حذف کن و مدل را دوباره تست بگیر.
- **تغییر تنظیمات معماری و منظم‌کننده:** معماری شبکه را ساده‌تر یا پیچیده‌تر کن و پارامترهای منظم‌کننده را تغییر بده تا عملکرد مدل بهتر شود.


In [ ]:
mlp_trials = [
    MLPRegressor(hidden_layer_sizes=(8,), alpha=0.1, max_iter=100, random_state=42, early_stopping=True),
    MLPRegressor(hidden_layer_sizes=(32,), alpha=0.01, max_iter=150, random_state=42, early_stopping=True),
    MLPRegressor(hidden_layer_sizes=(32, 16), alpha=0.1, max_iter=200, random_state=42, early_stopping=True)
]
for mlp in mlp_trials:
    mlp.fit(X_sub, y_sub)
    y_pred = mlp.predict(X_val_processed)
    print_scores(f'MLP alpha={mlp.alpha}, layers={mlp.hidden_layer_sizes}', y_val, y_pred)


<div dir="rtl">
تبدیل ستون price به لگاریتم (log) برای شبکه عصبی فقط: گاهی در داده با پراکندگی زیاد، شبکه عصبی با هدف لگاریتمی نتیجه طبیعی‌تر می‌دهد:

In [ ]:
import numpy as np
y_sub_log = np.log1p(y_sub)  # log(1 + price)
y_val_log = np.log1p(y_val)

mlp_log = MLPRegressor(hidden_layer_sizes=(16,), alpha=0.01, max_iter=100, random_state=42, early_stopping=True)
mlp_log.fit(X_sub, y_sub_log)
y_val_pred_log = mlp_log.predict(X_val_processed)
print_scores('MLP (log target)', y_val_log, y_val_pred_log)


<div dir="rtl">

## تحلیل نتایج مدل MLP و نکات کلیدی

- مدل‌های MLP با هدف اصلی (پیش‌بینی قیمت واقعی) نتوانستند یادگیری انجام دهند و مقدار خطا و R² تقریباً ثابت و منفی باقی ماند؛ یعنی مدل عملاً فقط میانگین را پیش‌بینی کرده است و هیچ ارتباط موثری بین ویژگی‌ها و قیمت پیدا نکرده است.

- مدل MLP با هدف لگاریتمی (log1p(price)) کمی بهتر عمل کرده و R² را تا حدود 0.13 رسانده، اما هنوز بسیار ضعیف است و تنها بخش کوچکی از واریانس داده را توضیح داده.

---

## دلایل ضعف مدل MLP

- شبکه عصبی برای داده‌های رگرسیون، به داده بسیار تمیز، مقیاس‌بندی و حذف داده‌های پرت نیاز دارد.
- تعداد زیاد فیچرهای دسته‌ای و OneHot شده باعث شده مدل قادر نباشد روابط موثر بسازد و پراکندگی داده‌ها مدل را گیج کرده است.
- برای داده‌هایی با دامنه بسیار گسترده قیمت، مدل‌های رگرسیونی مثل درخت تصمیم و KNN معمولاً مناسب‌تر هستند.

---

## چرا تبدیل لگاریتمی (log-transform) تا حدی کار کرد؟

- تبدیل هدف به log باعث شد تاثیر نمونه‌هایی با قیمت خیلی بزرگ محدود شود و مدل بتواند کمی بهتر واریانس داده را توضیح دهد.
- با این حال، عدد R² هنوز پایین است و کاربرد عملی ندارد.

---

## پیشنهاد عملی

- مدل‌های KNN و Decision Tree با بهینه‌سازی پارامترها (که قبلاً تست کردی) بهترین عملکرد را دارند و برای پیش‌بینی قیمت خانه توصیه می‌شوند.
- اگر می‌خواهی شبکه عصبی را امتحان کنی: فقط روی داده بسیار تمیز، با حذف فیچرهای کم‌اهمیت و حذف قیمت‌های پرت تست بگیر.
- مدل‌های ensemble مثل Random Forest یا GradientBoosting معمولاً نتیجه خیلی بهتری می‌دهند.
- برای ارزیابی نهایی، نتایج مدل‌های درخت تصمیم و KNN را روی داده تست بررسی و از آن‌ها در خروجی نهایی استفاده کن.

</div>


In [ ]:

y_test_pred = dt_best.predict(X_test_processed)

In [ ]:
test_df = X_test.copy()
test_df["actual_price"] = y_test
test_df["predicted_price"] = y_test_pred
test_df["price_ratio"] = test_df["actual_price"] / test_df["predicted_price"]


In [ ]:
def price_category(row):
    if row["price_ratio"] > 1.2:
        return "over-value"
    elif row["price_ratio"] < 0.8:
        return "under-value"
    else:
        return "normal"

test_df["price_category"] = test_df.apply(price_category, axis=1)


۴. تحلیل آماری
شمار هر دسته را ببین:

In [ ]:
print(test_df["price_category"].value_counts())


In [ ]:
print(test_df.groupby("price_category")[["building_size", "neighborhood_avg_price", "actual_price"]].mean())


In [ ]:
test_df["price_category"].dropna().unique()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


test_df = X_test.copy()
test_df["actual_price"] = y_test
test_df["predicted_price"] = y_test_pred
test_df["price_ratio"] = test_df["actual_price"] / test_df["predicted_price"]

def price_category(row):
    if row["price_ratio"] > 1.2:
        return "over-value"
    elif row["price_ratio"] < 0.8:
        return "under-value"
    else:
        return "normal"

test_df["price_category"] = test_df.apply(price_category, axis=1)


In [ ]:
category_counts = test_df["price_category"].value_counts().sort_index()
category_counts.plot(kind="bar", color=["red", "green", "blue"])
plt.title("tedad khane ha bar asas daste gheymat")
plt.ylabel("tedad khane")
plt.xlabel("daste gheymat")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# میانگین متراژ و قیمت واقعی در هر دسته
print(test_df.groupby("price_category")[["building_size", "actual_price"]].mean())

# پنج محله با بیشترین آگهی در هر دسته
print(test_df.groupby("price_category")["neighborhood_slug"].value_counts().groupby(level=0).head(5))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.boxplot(data=test_df, x="price_category", y="building_size", palette=["red", "green", "blue"])
plt.title("tedad khane ha bar asas daste gheymat")
plt.ylabel("metrazh bana( metr moraba)")
plt.xlabel("daste gheymat")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=test_df, x="price_category", y="actual_price", palette=["red", "green", "blue"])
plt.title("tedad khane ha bar asas daste gheymat")
plt.ylabel("gheymat vaghe e")
plt.xlabel("daste gheymat")
plt.show()


In [ ]:
top_neigh = (
    test_df.groupby("price_category")["neighborhood_slug"]
    .value_counts()
    .rename("count")
    .reset_index()
)

for category in top_neigh['price_category'].unique():
    subset = top_neigh[top_neigh['price_category'] == category].nlargest(4, 'count')
    plt.bar(subset['neighborhood_slug'], subset['count'])
    plt.title(f"tedad khane dar 4 mahale por tekrar{category}")
    plt.ylabel("tedad khane")
    plt.xlabel("mahale")
    plt.show()
